# Data Science - Atrial Fibrillation

In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier

from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

In [ ]:
# read the excel file, remove the labels from the train set and use the labels as a control set.
# then show the shape of the data
df = pd.read_excel("data/Preprocessed_AFData.xlsx")

X = df.drop(columns=["Control"])
y = df["Control"]

In [ ]:
# remove features that do not hold much information:
from sklearn.decomposition import PCA
# Apply PCA to retain 99% variance
pca = PCA(n_components=0.99)  # Keeps enough components to explain 99% of variance
X_pca = pca.fit_transform(X_scaled)

# Get the number of retained features
num_original_features = X.shape[1]
num_retained_features = X_pca.shape[1]

# Find the importance of each original feature
feature_importances = np.abs(pca.components_).sum(axis=0)
sorted_indices = np.argsort(feature_importances)[::-1]

# Identify removed features
removed_features = X.columns.difference(X.columns[sorted_indices[:num_retained_features]])

# Print results
print(f"Original number of features: {num_original_features}")
print(f"Number of features retained: {num_retained_features}")
print(f"Removed features: {list(removed_features)}")

In [ ]:
X_reduced.shape

In [ ]:
y.shape

## Model Options

In [ ]:
# all 6 models with different options to deal with imbalanced datasets.

# The class_weight='balanced' option adjusts the weights of the classes inversely proportional to their frequencies.
models_class_weight = {
    "Logistic Regression": LogisticRegression(class_weight='balanced', max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(class_weight='balanced'),
    "Random Forest": RandomForestClassifier(class_weight='balanced'),
    "SVM": SVC(class_weight='balanced'),
}

# This dictionary contains models that do not support class weighting but will be used with SMOTE (Synthetic Minority Over-sampling Technique)
# to balance the dataset.
models_smote_only = {
    "Naive Bayes": GaussianNB(),
    "KNN": KNeighborsClassifier(),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss')
}

# Same models as models_class_weight, but they will also be used with SMOTE.
models_smote_plus_weight = {
    "LogReg + SMOTE": LogisticRegression(class_weight='balanced', max_iter=1000),
    "DT + SMOTE": DecisionTreeClassifier(class_weight='balanced'),
    "RF + SMOTE": RandomForestClassifier(class_weight='balanced'),
    "SVM + SMOTE": SVC(class_weight='balanced')
}

## Helper Functions

In [ ]:
# cross validation, using 5 folds.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# train and evaluate models using the pipeline, then show feature importances for each model.
def evaluate_model(model_name, pipeline):
    print(f"\n=== {model_name} ===")
    all_y_true = []
    all_y_pred = []

    for fold_idx, (train_idx, test_idx) in enumerate(cv.split(X, y), 1):
        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        # Reduce the training set while maintaining class balance
        X_train, _, y_train, _ = train_test_split(
            X_train, y_train, train_size=0.1, stratify=y_train, random_state=42
        )


        pipeline.fit(X_train, y_train)
        y_pred = pipeline.predict(X_test)

        all_y_true.extend(y_test)
        all_y_pred.extend(y_pred)

    print(f"\nOverall classification report for {model_name}:")
    print(classification_report(all_y_true, all_y_pred, digits=3))

    model = pipeline.named_steps["classifier"]

    if hasattr(model, "feature_importances_"):
        importances = model.feature_importances_
    elif hasattr(model, "coef_"):
        importances = np.abs(model.coef_[0])
    else:
        print(f"Feature importance not available for {model_name}")

    importance_df = pd.DataFrame({
        "Feature": X.columns,
        "Importance": importances
    }).sort_values(by="Importance", ascending=False)

    print("\nTop Feature Importances:")
    print(importance_df.to_string(index=False))

## Train + Test Models

In [ ]:
# build pipelines for each model, using the method for imbalanced datasets explained above.
# then train and test the models using the helper function above.
print("\n=== Class Weight Models ===")
for name, model in models_class_weight.items():
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('classifier', model)
    ])
    evaluate_model(name, pipe)

print("\n=== SMOTE Only Models ===")
for name, model in models_smote_only.items():
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('smote', SMOTE(random_state=42)),
        ('classifier', model)
    ])
    evaluate_model(name, pipe)

print("\n=== SMOTE + Class Weight Models ===")
for name, model in models_smote_plus_weight.items():
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('smote', SMOTE(random_state=42)),
        ('classifier', model)
    ])
    evaluate_model(name, pipe)